# Time Series Analysis for Portfolio Management
## Task 1: Data Extraction, Cleaning, and EDA

This notebook performs comprehensive exploratory data analysis on three assets:
- **TSLA**: Tesla, Inc. (High growth stock)
- **BND**: Vanguard Total Bond Market ETF (Bond fund)
- **SPY**: SPDR S&P 500 ETF (Market index)

**Data Range**: January 1, 2015 to June 30, 2026

In [ ]:
# Import required libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")

## 1. Data Extraction
### Using yfinance to fetch historical data

In [ ]:
# Define tickers and date range
TICKERS = ['TSLA', 'BND', 'SPY']
START_DATE = '2015-01-01'
END_DATE = '2026-06-30'

# Fetch data using yfinance
print(f"Fetching data for {TICKERS} from {START_DATE} to {END_DATE}...")

# Download historical data
raw_data = yf.download(TICKERS, start=START_DATE, end=END_DATE, progress=True)

# Extract closing prices
if len(TICKERS) > 1:
    prices = raw_data['Close'][TICKERS]
else:
    prices = raw_data[['Close']]
    prices.columns = TICKERS

# Display first few rows
print("\n=== First 5 Rows of Closing Prices ===")
display(prices.head())

print("\n=== Last 5 Rows of Closing Prices ===")
display(prices.tail())

print(f"\nData Shape: {prices.shape}")
print(f"Date Range: {prices.index.min()} to {prices.index.max()}")
print(f"Total Trading Days: {len(prices)}")

In [ ]:
# Save raw data
prices.to_csv('../data/raw/stock_prices.csv')
print("Raw data saved to data/raw/stock_prices.csv")

## 2. Data Cleaning
### Checking data types, missing values, and basic statistics

In [ ]:
# Check data types
print("=== Data Types ===")
print(prices.dtypes)
print()

# Check for missing values
print("=== Missing Values ===")
missing = prices.isnull().sum()
print(missing)
print(f"\nTotal Missing: {missing.sum()}")
print(f"Missing Percentage: {(missing.sum() / (prices.shape[0] * prices.shape[1])) * 100:.2f}%")

In [ ]:
# Handle missing values if any
if prices.isnull().any().any():
    print("\nHandling missing values...")
    # Forward fill first (use previous day's price)
    prices = prices.fillna(method='ffill')
    # Backward fill any remaining (for start of series)
    prices = prices.fillna(method='bfill')
    print(f"Missing values after cleaning: {prices.isnull().sum().sum()}")
else:
    print("No missing values found in the dataset!")

# Check for duplicates
duplicates = prices.index.duplicated().sum()
print(f"\nDuplicate dates: {duplicates}")

if duplicates > 0:
    prices = prices[~prices.index.duplicated()]
    print(f"Removed {duplicates} duplicate entries")

In [ ]:
# Basic statistics summary
print("=== Descriptive Statistics ===")
display(prices.describe().round(2))

In [ ]:
# Additional statistics
stats_summary = pd.DataFrame({
    'Min': prices.min(),
    'Max': prices.max(),
    'Mean': prices.mean(),
    'Median': prices.median(),
    'Std Dev': prices.std(),
    'Skewness': prices.skew(),
    'Kurtosis': prices.kurtosis()
})

print("=== Statistical Summary ===")
display(stats_summary.round(4))

## 3. Exploratory Data Analysis (EDA) Visualizations

### 3.1 Closing Prices Over Time

In [ ]:
# Visualization 1: Closing Prices Over Time
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

colors = {'TSLA': '#E31937', 'BND': '#003366', 'SPY': '#FF6600'}

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    prices[ticker].plot(ax=ax, color=colors[ticker], linewidth=1.5)
    ax.set_title(f'{ticker} Closing Price Over Time', fontsize=14, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.grid(True, alpha=0.3)
    
    # Add annotations for min/max
    min_val = prices[ticker].min()
    max_val = prices[ticker].max()
    min_date = prices[ticker].idxmin()
    max_date = prices[ticker].idxmax()
    
    ax.annotate(f'Low: ${min_val:.2f}', xy=(min_date, min_val), 
                xytext=(min_date, min_val * 0.8),
                arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)
    ax.annotate(f'High: ${max_val:.2f}', xy=(max_date, max_val),
                xytext=(max_date, max_val * 1.1),
                arrowprops=dict(arrowstyle='->', color='green'), fontsize=10)

axes[2].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../data/processed/closing_prices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/closing_prices.png")

### 3.2 Daily Percentage Change (Returns)

In [ ]:
# Calculate daily percentage returns
returns = prices.pct_change().dropna()

print("=== Daily Returns Statistics ===")
print(f"Shape: {returns.shape}")
display(returns.describe().round(4))

In [ ]:
# Visualization 2: Daily Returns Over Time
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    returns[ticker].plot(ax=ax, color=colors[ticker], linewidth=0.8)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_title(f'{ticker} Daily Percentage Change', fontsize=14, fontweight='bold')
    ax.set_ylabel('Daily Return')
    ax.grid(True, alpha=0.3)
    
    # Highlight extreme days (> 2 std from mean)
    mean_ret = returns[ticker].mean()
    std_ret = returns[ticker].std()
    ax.axhline(y=mean_ret + 2*std_ret, color='red', linestyle='--', alpha=0.5)
    ax.axhline(y=mean_ret - 2*std_ret, color='red', linestyle='--', alpha=0.5)

axes[2].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../data/processed/daily_returns.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/daily_returns.png")

In [ ]:
# Daily Returns Distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    returns[ticker].hist(ax=ax, bins=50, color=colors[ticker], alpha=0.7, edgecolor='black')
    ax.axvline(returns[ticker].mean(), color='red', linestyle='--', label=f'Mean: {returns[ticker].mean():.4f}')
    ax.axvline(returns[ticker].median(), color='blue', linestyle='--', label=f'Median: {returns[ticker].median():.4f}')
    ax.set_title(f'{ticker} Return Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/returns_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/returns_distribution.png")

### 3.3 Rolling Mean and Standard Deviation (Volatility)

In [ ]:
# Calculate rolling statistics
ROLLING_WINDOW = 30  # 30 trading days (~6 weeks)

rolling_mean = prices.rolling(window=ROLLING_WINDOW).mean()
rolling_std = prices.rolling(window=ROLLING_WINDOW).std()

print(f"Rolling window size: {ROLLING_WINDOW} days")
print(f"Rolling mean shape: {rolling_mean.shape}")
print(f"Rolling std shape: {rolling_std.shape}")

In [ ]:
# Visualization 3: Rolling Mean and Volatility
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

for i, ticker in enumerate(TICKERS):
    # Rolling Mean
    ax1 = axes[i, 0]
    prices[ticker].plot(ax=ax1, color=colors[ticker], linewidth=1, alpha=0.5, label='Actual')
    rolling_mean[ticker].plot(ax=ax1, color='blue', linewidth=2, label=f'{ROLLING_WINDOW}-day MA')
    ax1.set_title(f'{ticker}: Price vs Rolling Mean', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Price ($)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Rolling Volatility (Standard Deviation)
    ax2 = axes[i, 1]
    rolling_std[ticker].plot(ax=ax2, color='red', linewidth=1.5)
    ax2.fill_between(rolling_std.index, 0, rolling_std[ticker], alpha=0.3, color='red')
    ax2.set_title(f'{ticker}: Rolling Volatility (Std Dev)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Standard Deviation ($)')
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/rolling_statistics.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/rolling_statistics.png")

In [ ]:
# Annualized Rolling Volatility
rolling_vol_annual = returns.rolling(window=ROLLING_WINDOW).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 6))

for ticker in TICKERS:
    rolling_vol_annual[ticker].plot(ax=ax, label=ticker, linewidth=1.5)

ax.set_title('Annualized Rolling Volatility (30-day window)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/annualized_volatility.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/annualized_volatility.png")

## 4. Outlier Detection
### Identifying days with unusually high or low returns

In [ ]:
# Outlier Detection using Z-score method
def detect_outliers_zscore(series, threshold=3):
    """Detect outliers using Z-score method."""
    z_scores = np.abs((series - series.mean()) / series.std())
    return z_scores > threshold

def detect_outliers_iqr(series, k=1.5):
    """Detect outliers using IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return (series < lower) | (series > upper)

# Detect outliers using Z-score
outlier_results = {}
THRESHOLD = 3  # 3 standard deviations

for ticker in TICKERS:
    z_outliers = detect_outliers_zscore(returns[ticker], THRESHOLD)
    outlier_results[ticker] = {
        'z_score': returns[ticker][z_outliers],
        'count': z_outliers.sum(),
        'dates': returns[ticker][z_outliers].index.tolist()
    }

    print(f"\n=== {ticker} Outliers (|z| > {THRESHOLD}) ===")
    print(f"Total outliers: {outlier_results[ticker]['count']}")
    print(f"Highest positive return: {returns[ticker].max():.4%} on {returns[ticker].idxmax().date()}")
    print(f"Most negative return: {returns[ticker].min():.4%} on {returns[ticker].idxmin().date()}")
    
    # Show top 5 most extreme days
    extreme_days = outlier_results[ticker]['z_score'].abs().sort_values(ascending=False).head(5)
    print(f"\nTop 5 most extreme days:")
    for date in extreme_days.index:
        ret = returns.loc[date, ticker]
        print(f"  {date.date()}: {ret:+.4%}")

In [ ]:
# Visualization: Outliers highlighted
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    
    # Plot all returns
    ax.scatter(returns.index, returns[ticker], c=['red' if x else 'black' for x in detect_outliers_zscore(returns[ticker])], 
               alpha=0.6, s=20)
    
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    ax.axhline(y=returns[ticker].mean() + 3*returns[ticker].std(), color='red', linestyle='--', alpha=0.7, label='3 Std Dev')
    ax.axhline(y=returns[ticker].mean() - 3*returns[ticker].std(), color='red', linestyle='--', alpha=0.7)
    
    ax.set_title(f'{ticker} Daily Returns with Outliers Highlighted', fontsize=14, fontweight='bold')
    ax.set_ylabel('Daily Return')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/outlier_detection.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/outlier_detection.png")

In [ ]:
# Save outlier analysis
outlier_summary = pd.DataFrame({
    'Ticker': TICKERS,
    'Outlier Count': [outlier_results[t]['count'] for t in TICKERS],
    'Max Positive Return': [returns[t].max() for t in TICKERS],
    'Max Positive Date': [returns[t].idxmax().date() for t in TICKERS],
    'Max Negative Return': [returns[t].min() for t in TICKERS],
    'Max Negative Date': [returns[t].idxmin().date() for t in TICKERS]
})

outlier_summary.to_csv('../data/processed/outlier_summary.csv', index=False)
display(outlier_summary)
print("\nOutlier summary saved to data/processed/outlier_summary.csv")

## 5. Stationarity Testing
### Augmented Dickey-Fuller (ADF) Test

The ADF test checks for stationarity in time series data.
- **Null Hypothesis (H0)**: The series has a unit root (non-stationary)
- **Alternative Hypothesis (H1)**: The series does not have a unit root (stationary)

Interpretation:
- If p-value < 0.05, reject H0 → Series is stationary
- If p-value >= 0.05, fail to reject H0 → Series is non-stationary

In [ ]:
def adf_test(series, name):
    """
    Perform Augmented Dickey-Fuller test and print results.
    """
    result = adfuller(series.dropna(), autolag='AIC')
    
    output = {
        'Series': name,
        'ADF Statistic': result[0],
        'p-value': result[1],
        'Used Lags': result[2],
        'Observations': result[3],
        'Critical Value (1%)': result[4]['1%'],
        'Critical Value (5%)': result[4]['5%'],
        'Critical Value (10%)': result[4]['10%']
    }
    
    # Determine stationarity
    if result[1] < 0.05:
        output['Is Stationary'] = True
        output['Conclusion'] = 'Stationary (reject H0)'
    else:
        output['Is Stationary'] = False
        output['Conclusion'] = 'Non-stationary (fail to reject H0)'
    
    return output

# Test stationarity for closing prices
print("=" * 60)
print("STATIONARITY TEST RESULTS")
print("=" * 60)

# Test on Closing Prices
print("\n### Testing Closing Prices ###")
adf_results_prices = []

for ticker in TICKERS:
    result = adf_test(prices[ticker], f'{ticker} Price')
    adf_results_prices.append(result)
    
    print(f"\n{ticker} Closing Prices:")
    print(f"  ADF Statistic: {result['ADF Statistic']:.4f}")
    print(f"  p-value: {result['p-value']:.4f}")
    print(f"  Critical Values: 1%: {result['Critical Value (1%)']:.4f}, "
          f"5%: {result['Critical Value (5%)']:.4f}, 10%: {result['Critical Value (10%)']:.4f}")
    print(f"  => CONCLUSION: {result['Conclusion']}")

# Test on Daily Returns
print("\n### Testing Daily Returns ###")
adf_results_returns = []

for ticker in TICKERS:
    result = adf_test(returns[ticker], f'{ticker} Returns')
    adf_results_returns.append(result)
    
    print(f"\n{ticker} Daily Returns:")
    print(f"  ADF Statistic: {result['ADF Statistic']:.4f}")
    print(f"  p-value: {result['p-value']:.4f}")
    print(f"  Critical Values: 1%: {result['Critical Value (1%)']:.4f}, "
          f"5%: {result['Critical Value (5%)']:.4f}, 10%: {result['Critical Value (10%)']:.4f}")
    print(f"  => CONCLUSION: {result['Conclusion']}")

In [ ]:
# Summary DataFrame
print("\n### Stationarity Test Summary ###")

adf_summary = pd.DataFrame(adf_results_prices + adf_results_returns)
adf_summary['Critical Value (10%)'] = adf_summary['Critical Value (10%)'].astype(float)
adf_summary.to_csv('../data/processed/stationarity_tests.csv', index=False)

display(adf_summary[['Series', 'ADF Statistic', 'p-value', 'Conclusion']])
print("\nStationarity results saved to data/processed/stationarity_tests.csv")

### Interpretation of Stationarity Results

1. **Closing Prices**: Typically non-stationary as they have trends and drift. This means we cannot directly use them for many time series models.

2. **Daily Returns**: Typically stationary, making them suitable for modeling. The p-value is usually very small, rejecting the null hypothesis of non-stationarity.

**Implication for Modeling**: We should use returns (differenced prices) for ARIMA models, as they satisfy stationarity requirements.

## 6. Risk Metrics
### Value at Risk (VaR) and Sharpe Ratio

In [ ]:
# Risk Metrics Calculation
TRADING_DAYS = 252
RISK_FREE_RATE = 0.02  # 2% annual risk-free rate

def calculate_var(returns, confidence=0.95, method='parametric'):
    """
    Calculate Value at Risk (VaR).
    
    Parameters:
    - returns: Series of daily returns
    - confidence: Confidence level (e.g., 0.95 for 95%)
    - method: 'parametric', 'historical', or 'monte_carlo'
    
    Returns:
    - VaR value (as a positive number indicating potential loss)
    """
    if method == 'parametric':
        mu = returns.mean()
        sigma = returns.std()
        z_score = stats.norm.ppf(1 - confidence)
        var = -(mu + z_score * sigma)
    elif method == 'historical':
        var = -np.percentile(returns, (1 - confidence) * 100)
    elif method == 'monte_carlo':
        mu = returns.mean()
        sigma = returns.std()
        simulated = np.random.normal(mu, sigma, 10000)
        var = -np.percentile(simulated, (1 - confidence) * 100)
    return var

def calculate_expected_shortfall(returns, confidence=0.95):
    """Calculate Expected Shortfall (CVaR)."""
    var = calculate_var(returns, confidence=confidence)
    es = -returns[returns <= -var].mean() if len(returns[returns <= -var]) > 0 else var
    return es

def calculate_sharpe_ratio(returns, risk_free_rate=RISK_FREE_RATE, periods=TRADING_DAYS):
    """Calculate annualized Sharpe Ratio."""
    excess_returns = returns - risk_free_rate / periods
    annualized_return = returns.mean() * periods
    annualized_vol = returns.std() * np.sqrt(periods)
    sharpe = (annualized_return - risk_free_rate) / annualized_vol
    return sharpe

# Calculate risk metrics for each asset
print("=" * 60)
print("RISK METRICS SUMMARY")
print("=" * 60)

risk_metrics = []

for ticker in TICKERS:
    ret = returns[ticker]
    
    # VaR calculations
    var_95_param = calculate_var(ret, 0.95, 'parametric')
    var_95_hist = calculate_var(ret, 0.95, 'historical')
    var_99 = calculate_var(ret, 0.99, 'parametric')
    
    # Expected Shortfall
    es_95 = calculate_expected_shortfall(ret, 0.95)
    
    # Sharpe Ratio
    sharpe = calculate_sharpe_ratio(ret)
    
    # Annualized metrics
    annual_return = ret.mean() * TRADING_DAYS
    annual_vol = ret.std() * np.sqrt(TRADING_DAYS)
    
    # Maximum drawdown
    cumulative = (1 + ret).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min()
    
    metrics = {
        'Ticker': ticker,
        'Annualized Return': annual_return,
        'Annualized Volatility': annual_vol,
        'Sharpe Ratio': sharpe,
        f'VaR 95% (Parametric)': var_95_param,
        f'VaR 95% (Historical)': var_95_hist,
        f'VaR 99% (Parametric)': var_99,
        'Expected Shortfall 95%': es_95,
        'Max Drawdown': max_dd
    }
    
    risk_metrics.append(metrics)
    
    print(f"\n### {ticker} Risk Metrics ###")
    print(f"  Annualized Return: {annual_return:.4%}")
    print(f"  Annualized Volatility: {annual_vol:.4%}")
    print(f"  Sharpe Ratio: {sharpe:.4f}")
    print(f"  VaR (95% Parametric): {var_95_param:.4%}")
    print(f"  VaR (95% Historical): {var_95_hist:.4%}")
    print(f"  VaR (99% Parametric): {var_99:.4%}")
    print(f"  Expected Shortfall (95%): {es_95:.4%}")
    print(f"  Maximum Drawdown: {max_dd:.4%}")

In [ ]:
# Create risk metrics summary table
risk_df = pd.DataFrame(risk_metrics)
risk_df.to_csv('../data/processed/risk_metrics.csv', index=False)

print("\n### Risk Metrics Summary Table ###")
display(risk_df.round(4))
print("\nRisk metrics saved to data/processed/risk_metrics.csv")

In [ ]:
# Visualization: VaR Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sharpe Ratio Comparison
sharpe_values = [m['Sharpe Ratio'] for m in risk_metrics]
axes[0].bar(TICKERS, sharpe_values, color=[colors[t] for t in TICKERS], edgecolor='black')
axes[0].axhline(y=1, color='green', linestyle='--', label='Good (>1)')
axes[0].axhline(y=0, color='red', linestyle='--', label='Neutral')
axes[0].set_title('Sharpe Ratio Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VaR Comparison
var_values = [m['VaR 95% (Parametric)'] for m in risk_metrics]
axes[1].bar(TICKERS, [v * 100 for v in var_values], color=[colors[t] for t in TICKERS], edgecolor='black')
axes[1].set_title('Value at Risk (95%) Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('VaR 95% (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/risk_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/risk_metrics_comparison.png")

## Summary and Insights

### Key Findings:

1. **Data Quality**: The dataset spans from {START_DATE} to {END_DATE} with minimal/no missing values.

2. **Price Trends**: 
   - TSLA shows highest volatility and dramatic price swings
   - BND shows stable, gradual appreciation typical of bond funds
   - SPY follows broad market trends with moderate volatility

3. **Stationarity**: 
   - Closing prices are non-stationary (require differencing)
   - Daily returns are stationary (suitable for modeling)

4. **Risk Analysis**:
   - TSLA has highest VaR (most potential loss risk)
   - BND has lowest volatility and best risk-adjusted returns for conservative investors
   - SPY provides balanced risk-return profile

### Next Steps:
   - Proceed to Task 2: Implement forecasting models (ARIMA/SARIMA/LSTM)
   - Use returns data for modeling (stationary)
   - Consider ensemble approaches for improved predictions

In [ ]:
# Save processed data
returns.to_csv('../data/processed/daily_returns.csv')
rolling_mean.to_csv('../data/processed/rolling_mean.csv')
rolling_std.to_csv('../data/processed/rolling_std.csv')

print("All processed data saved to data/processed/")
print("\nEDA Notebook Complete!")